In [1]:
!pip install -q transformers datasets sentencepiece accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": "/content/drive/MyDrive/NLP project/train.jsonl"}
)

print(dataset)
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 40
    })
})
{'input': '이수민 교수님께 자연어처리 수업 출결 관련 메일 작성. 다음 주 몸살로 인해 수업에 참석하기 어려울 것 같습니다.', 'output': '안녕하세요, 이수민 교수님.\n저는 교수님의 자연어처리 수업을 수강 중인 OOO입니다.\n\n다음 주 몸살 증상으로 인해 부득이하게 수업에 참석하기 어려울 것 같아 메일 드립니다. 갑작스럽게 연락드리게 되어 죄송합니다. 혹시 출결과 관련하여 제가 추가로 확인하거나 제출해야 할 사항이 있다면 알려주시면 감사하겠습니다.\n\n감사합니다.\nOOO 드림'}


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "KETI-AIR/ke-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/597 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.47M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/189 [00:00<?, ?it/s]

In [ ]:
max_input_length = 128
max_target_length = 256

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input"],
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["output"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_dataset = dataset["train"].map(preprocess_function, batched=True)

print(tokenized_dataset[0].keys())

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

dict_keys(['input', 'output', 'input_ids', 'attention_mask', 'labels'])


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./email_model",
    num_train_epochs=20,
    per_device_train_batch_size=4,
    learning_rate=5e-5
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./email_model",
    num_train_epochs=20,
    per_device_train_batch_size=4,
    learning_rate=5e-5
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

In [ ]:
trainer.save_model("./email_model_final")
tokenizer.save_pretrained("./email_model_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./email_model_final/tokenizer_config.json',
 './email_model_final/tokenizer.json')

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def generate_email(prompt, max_new_tokens=256):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
test_prompt = "이수민 교수님께 자연어처리 수업 출결 관련 메일 작성. 다음 주 병원 진료로 인해 수업에 참석하기 어려울 것 같습니다."

result = generate_email(test_prompt)
print(result)

Facilitated     
